# 🧠 Neuromorphic SNN for Energy-Efficient Anomaly Detection
**SNN-Abnormality-Detection**

This notebook runs the full experiment on Google Colab's T4 GPU.
Runtime → Change runtime type → T4 GPU (free tier)

**Architecture**: Spiking Neural Network (SNN) Autoencoder using SpikingJelly
- Neurons: Leaky Integrate-and-Fire (LIF) with surrogate gradient training
- Encoding: Rate-coded spike trains over T=8 timesteps  
- Datasets: Thyroid, Arrhythmia, Cardio (ODDS benchmarks)
- Comparison: SNN vs ANN baseline on F1, AUC, FLOPs, SynOps, latency

In [ ]:
# ── STEP 1: Clone your repo ──────────────────────────────────────────────────
# Replace with your actual GitHub URL once you push
!git clone https://github.com/samiurk70/SNN-Abnormality-Detection.git
%cd SNN-Abnormality-Detection

In [ ]:
# ── STEP 2: Install dependencies ─────────────────────────────────────────────
%pip install spikingjelly pyod tqdm --quiet
print('Dependencies installed!')

In [ ]:
# ── STEP 3: Verify GPU ───────────────────────────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── STEP 4: LIF Neuron Demo ──────────────────────────────────────────────────
# Run this to understand what happens inside each neuron before training
import sys
sys.path.insert(0, '.')

import os
os.makedirs('results', exist_ok=True)

from src.neuromorphic.lif_neuron import demo_lif
demo_lif()

from IPython.display import Image
Image('results/lif_demo.png')

In [ ]:
# ── STEP 5: Spike Encoding Demo ──────────────────────────────────────────────
from src.neuromorphic.spike_encoder import RateEncoder
import torch

sample_features = torch.tensor([[0.9, 0.1, 0.5, 0.0, 1.0]])
encoder = RateEncoder(T=10)
spikes = encoder.encode(sample_features)

print('Input features:    ', sample_features.numpy())
print('Spike train (T=10):')
print(spikes[:, 0, :].numpy())  # shape (T, features)
print('Mean spike rates:  ', spikes.mean(dim=0).numpy())
print('→ Should be close to input values')

In [ ]:
# ── STEP 6: Train on ONE dataset (thyroid) ───────────────────────────────────
!python train.py --dataset thyroid --epochs 50 --T 8 --device cuda

In [ ]:
# ── STEP 7: View training loss curve ─────────────────────────────────────────
from IPython.display import Image
Image('results/thyroid/training_loss.png')

In [ ]:
# ── STEP 8: View error distribution ─────────────────────────────────────────
Image('results/thyroid/error_distribution.png')

In [ ]:
# ── STEP 9: Run ALL three datasets ───────────────────────────────────────────
# This takes ~5-10 minutes on GPU, 30-60 min on CPU
!python train.py --dataset all --epochs 50 --T 8 --device cuda

In [ ]:
# ── STEP 10: Final comparison figure ─────────────────────────────────────────
Image('results/comparison_figure.png')

In [ ]:
# ── STEP 11: Save results to Google Drive (optional) ─────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree('results', '/content/drive/MyDrive/SNN_Results', dirs_exist_ok=True)
print('Results saved to Google Drive!')